In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

# **_Silver Transformation
# Trimming_**

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType,StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

# **_Cleaning up customer_id column_**

In [0]:
df = df.withColumn(
  "cid",
  F.when(
    col('cid').startswith('NAS'),
      F.substring(col('cid'),4,F.length(col('cid')))
  ).otherwise(col('cid'))
  )

# **_Birthdate Validation_**


In [0]:
df = df.withColumn(
    'bdate',
    F.when(col('bdate') > F.current_date(),None)
     .otherwise(col('bdate'))
)

# **_Gender Normalization_**


In [0]:
df.withColumn(
    'gen',
    F.when(F.upper(col('gen')).isin("F","FEMALE"),"Female")
     .when(F.upper(col('gen')).isin("M","MALE"),"Male")
     .otherwise('n/a')
)

# **_Renaming Columns_**


In [0]:
RENAME_MAP = {
    'cid':"customer_number",
    'bdate':"birth_date",
    'gen':"gender"
}
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)


In [0]:
df.limit(10).display()

# **_Writing into the silver table_**


In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('silver.erp_cust_az12')

In [0]:
%sql
SELECT * FROM workspace.silver.erp_cust_az12 LIMIT 10